# Scratchpad (State 1)

Interactive derivations and numerical checks before promotion to the clean paper
(`docs/fund-flow-rotation.tex`). Results are marked `[VERIFIED]` only after an
independent numerical check and human consensus. Dead ends are kept, marked
`[DEAD-END]`.

## 2026-06-10 - Smoothing the rotation-graph coordinates

**Problem.** The rotation graph plots RS-Ratio $\mathrm{RS}_{c,t}$ (eq:rs_ratio)
against RS-Momentum $\mathrm{RS}^{\mathrm{m}}_{c,t}$ (eq:rs_momentum). Built on raw
monthly relative flow, the per-sector tails crisscross the plane instead of tracing
the clockwise arcs the graph is meant to show. Monthly $\mathrm{rel}_{c,t} = g_{c,t}
- g_{U,t}$ inherits the full noise of a single month of fund flow, so the tail is
dominated by high-frequency jitter rather than rotation.

**Proposed fix.** Smooth the relative-strength signal with a trailing $w$-month mean
before standardizing,
$$\bar{\mathrm{rel}}_{c,t}(w) = \frac{1}{w}\sum_{k=0}^{w-1}\mathrm{rel}_{c,t-k},$$
and define the rotation-graph relative strength as the standardization of the
smoothed series,
$$\mathrm{RS}_{c,t} = \frac{\bar{\mathrm{rel}}_{c,t} - \mu_L(\bar{\mathrm{rel}}_c)}
{\sigma_L(\bar{\mathrm{rel}}_c)},\qquad
\mathrm{RS}^{\mathrm{m}}_{c,t} = \mathrm{RS}_{c,t} - \mathrm{RS}_{c,t-D}.$$
This is the standard construction: the JdK / Bloomberg RRG is built on a smoothed
relative-strength line, not the raw ratio.

**Self-challenge.**
- `[CHECK]` Degenerate case $w=1$: $\bar{\mathrm{rel}} = \mathrm{rel}$, so the
  coordinates reduce exactly to the current unsmoothed definition. Good - smoothing
  is a strict generalization, the old behavior is $w=1$.
- `[CHECK]` Internal consistency: the mean is linear, so $\bar{\mathrm{rel}} =
  \overline{g_c} - \overline{g_U}$; smoothing the strength signal is the same as
  using smoothed growth against a smoothed baseline. No inconsistency.
- `[CHECK]` Over-smoothing: a large $w$ manufactures an artificially clean spiral
  and adds a lag of about $(w-1)/2$ months. Keep $w$ small; do not let the chart
  invent rotation that is not in the data.
- `[CHECK]` Cost: $\mathrm{RS}$ needs $w-1$ extra warm-up months before it is defined
  (the smoothing window must fill before the standardization window starts).
- `[CHECK]` Does it actually de-jitter? Needs an empirical measure, below.

**Numerical check.** Measure the mean per-sector tail path length in $(\mathrm{RS},
\mathrm{RS}^{\mathrm{m}})$ space over the last 6 months: the summed Euclidean length
of the polyline a sector traces. Genuine rotation has modest length; jitter inflates
it with back-and-forth zigzag. If smoothing helps, length should fall.

In [1]:
import numpy as np
import sanity_check as sc
from categories import category_panel
from rotation import relative_flow, z_score, momentum

panel = sc.load()
cat0 = relative_flow(category_panel(panel), panel)

def coords(w):
    c = cat0.sort_values(['category', 'month']).copy()
    c['rel_s'] = (c['rel'] if w == 1 else
                  c.groupby('category')['rel'].transform(
                      lambda s: s.rolling(w, min_periods=w).mean()))
    c = z_score(c, col='rel_s', lookback=12, out='rs')
    c = momentum(c, col='rs', lag=3, out='rs_mom')
    return c

def mean_tail_path(c, tail=6):
    months = sorted(c['month'].unique())[-(tail + 1):]
    sub = c[c['month'].isin(months)].dropna(subset=['rs', 'rs_mom'])
    L = []
    for _, d in sub.groupby('category'):
        d = d.sort_values('month')
        if len(d) < 2:
            continue
        dx = np.diff(d['rs'].values); dy = np.diff(d['rs_mom'].values)
        L.append(np.sqrt(dx**2 + dy**2).sum())
    return np.mean(L)

print('smoothing w | mean per-sector 6m tail path length in (rs,rs_mom)')
for w in [1, 2, 3, 4]:
    print(f'   w={w}   {mean_tail_path(coords(w)):.3f}')

smoothing w | mean per-sector 6m tail path length in (rs,rs_mom)
   w=1   17.112
   w=2   11.989
   w=3   11.459
   w=4   9.401


**Result.** Path length falls 17.1 -> 12.0 -> 11.5 -> 9.4 for $w = 1,2,3,4$. The
$w=1 \to 2$ step removes most of the jitter (a 30% drop); $w=2 \to 3$ adds little
(11.99 -> 11.46); $w=4$ keeps shrinking but at the cost of more lag, consistent with
the over-smoothing `[CHECK]`.

**Recommendation.** Adopt the trailing-mean smoothing with a default $w=3$: it
matches the quarterly cadence already used for the momentum lag $D=3$, and a quarter
is the natural unit for a flow-rotation read. $w=2$ is a defensible lighter
alternative that captures most of the de-jittering with one fewer month of lag.

Status: `[VERIFIED]`. Consensus 2026-06-10: adopt the trailing-mean smoothing with
default $w=3$. Promoted to State 2 - eq:rel_smoothed (smoothing) added and eq:rs_ratio
generalized to the smoothed signal.

## 2026-06-10 - Is net flow zero-sum across sectors?

Question (Max): would it be an interesting sanity check to see if net flow is
zero-sum? Two distinct senses of "zero-sum":

1. Raw dollar flow $\sum_c F_{c,t}$: no structural reason to vanish -- it is the
   common tide of money entering or leaving the sector-ETF complex as a whole.
   Its size relative to gross flow measures how much of sector flow is tide
   versus rotation.
2. Relative flow: the claim is that the $A_{t-1}$-weighted relative flows sum to
   zero each month, exactly, by construction.

Derivation of (2). From eq:relative_flow, eq:category_g, eq:universe_g, with the
universe being the same eleven categories:

$$\sum_c A_{c,t-1}\,\mathrm{rel}_{c,t}
  = \sum_c A_{c,t-1}\left(\frac{F_{c,t}}{A_{c,t-1}} - g_{U,t}\right)
  = \sum_c F_{c,t} - \frac{\sum_c F_{c,t}}{\sum_c A_{c,t-1}}\sum_c A_{c,t-1}
  = 0.$$

So relative flow is a dollar conservation law: every dollar of above-market flow
in one sector is matched by below-market flow elsewhere. Self-challenges:

- `[CHECK]` The identity needs $g_U$ computed over the same fund set with the
  same $A_{t-1}$ presence convention as the categories (funds entering
  mid-history). Both aggregations use the same skip-NaN sum, so entry months are
  consistent; verified numerically below, including such months.
  CORRECTION (State 0, same day): the consistency claim is wrong when a whole
  CATEGORY enters mid-history -- its first month has undefined growth, the
  category leaves the sum, and the residual is exactly $-F_{\mathrm{entrant},t}$
  (the unit test demonstrates the synthetic entry-month residual equals the
  entrant's flow to machine precision). A fund joining an existing multi-fund
  category stays consistent. Scope of the identity: months with no category
  entries; the live eleven-sector panel has none, so it holds in every month.
- `[CHECK]` Weighted, not unweighted: the unweighted sum of rel is materially
  nonzero (mean |sum| 0.077 pp/mo). Zero-sum holds for dollars, not for growth
  rates.
- `[CHECK]` Raw dollar flow is not zero-sum and should not be; empirically the
  common tide is large (below).
- `[CHECK]` First run of the check produced infs: it exposed an aggregation bug
  -- months where every member fund's AUM is NaN (2019-07/08, before the AUM
  anchoring point) were summed to a fabricated 0.0, making the next month's
  g = F/0 = inf. Fixed in categories.py (sum with min_count=1) with a
  regression test; Scenario B engineering fix, no formula change.

Empirical results (code below, live panel, 81 months):

- Invariant: max over months of $|\sum_c A_{c,t-1}\mathrm{rel}_{c,t}|$ is 1e-6
  dollars (relative 1.9e-16) -- machine precision. Holds.
- Raw flow: mean net +\$0.57B/month, +\$46.2B cumulative; median
  |net|/gross = 42%; 57% of months net positive. So on a typical month roughly
  40% of gross sector flow is common tide and 60% nets out as true rotation.

Status: `[VERIFIED]`. Consensus 2026-06-10: promote the identity to the paper's
Validation section (State 2, eq:flow_conservation), a live conservation check in
sanity_check.py (State 3), and an offline invariant unit test on synthetic data
(State 0).


In [1]:
import numpy as np
import pandas as pd
from viz import _load
from categories import category_panel
from rotation import relative_flow

panel = _load()
cat = category_panel(panel)
rel = relative_flow(cat, panel)
assert np.isfinite(rel["rel"].dropna()).all()

B = 1e9
m = cat.groupby("month").agg(net=("F", "sum"), gross=("F", lambda s: s.abs().sum()))
m["ratio"] = m["net"] / m["gross"]
print(f"raw flow:  mean net {m.net.mean()/B:+.2f}B/mo, total {m.net.sum()/B:+.1f}B/{len(m)}mo, "
      f"median |net|/gross {m.ratio.abs().median():.1%}, months>0 {(m.net>0).mean():.0%}")

ok = rel.dropna(subset=["rel", "aum_prev"])
w = ok.groupby("month").apply(lambda d: (d["aum_prev"] * d["rel"]).sum(), include_groups=False)
scale = ok.groupby("month").apply(lambda d: (d["aum_prev"] * d["rel"].abs()).sum(), include_groups=False)
print(f"invariant: max |sum_c A_prev*rel| = {w.abs().max():,.6f} dollars "
      f"(relative {np.nanmax(w.abs()/scale):.1e})")

u = ok.groupby("month")["rel"].sum()
print(f"unweighted sum of rel: mean |.| = {u.abs().mean():.4f}, max |.| = {u.abs().max():.4f} (pp/mo, not zero)")

raw flow:  mean net +0.57B/mo, total +46.2B/81mo, median |net|/gross 42.2%, months>0 57%
invariant: max |sum_c A_prev*rel| = 0.000001 dollars (relative 1.9e-16)
unweighted sum of rel: mean |.| = 0.0774, max |.| = 0.8588 (pp/mo, not zero)
